# 0412 · Modelos base: referencia y lineal / logística

**Qué hace.** Valida los dos puntos de partida contra los que se comparan los ensambles (H1):

| Modelo | Clasificación (`clf`) | Regresión (`reg`) |
| --- | --- | --- |
| **Referencia** (sin modelo) | Prevalencia de incumplimiento del entrenamiento, igual para todas las guías | Media histórica de horas de su zona×franja (o la media del entrenamiento si no hay historia) |
| **Lineal** | Regresión logística | Regresión lineal |

Los dos usan las mismas variables y los mismos bloques de validación que los ensambles. No tienen hiperparámetros que buscar, así que este cuaderno es rápido.

**No depende de los otros cuadernos de modelos.** Solo necesita que 0411 se haya corrido. La prueba final no se toca aquí.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 153.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 125.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 20.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 109.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, pliegues_origen_movil, partes_del_bloque, marco_predicciones,
                                       construir_modelo, predecir, tabla_metricas)

In [0]:
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"

## 1. Cargar la preparación de 0411

In [0]:
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · preparado el {CFG['fecha_preparacion']}")

1,588,872 guías · versiones: ['internas', 'internas+clima'] · preparado el 2026-10-24T18:56:48.281707


## 2. Validación de origen móvil

En cada versión de variables, tarea y bloque: se arma el entrenamiento (antes del bloque) y la validación (el bloque), se calcula la referencia y se entrena el modelo lineal.

In [0]:
pred_ref, pred_lin = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            base = marco_predicciones(version, tarea, ini, Dva, yva)
            # Referencia: sin modelo, estimada solo con el entrenamiento
            ref = np.full(len(Dva), ytr.mean()) if tarea == "clf" else Dva["hist_h_zona_franja"].fillna(ytr.mean()).values
            pred_ref.append(base.assign(modelo="Referencia", pred=ref))
            # Lineal (regresión lineal o logística según la tarea)
            m = construir_modelo(tarea, "Lineal", numericas, CATEGORICAS, {}, SEMILLA).fit(Xtr, ytr)
            pred_lin.append(base.assign(modelo="Lineal", pred=predecir(m, Xva, tarea)))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred_ref, pred_lin = pd.concat(pred_ref, ignore_index=True), pd.concat(pred_lin, ignore_index=True)

internas · clf · 2026-04-01 listo (12 s)
internas · clf · 2026-05-01 listo (25 s)
internas · clf · 2026-06-01 listo (39 s)
internas · reg · 2026-04-01 listo (46 s)
internas · reg · 2026-05-01 listo (55 s)
internas · reg · 2026-06-01 listo (65 s)
internas+clima · clf · 2026-04-01 listo (74 s)
internas+clima · clf · 2026-05-01 listo (86 s)
internas+clima · clf · 2026-06-01 listo (100 s)
internas+clima · reg · 2026-04-01 listo (106 s)
internas+clima · reg · 2026-05-01 listo (120 s)
internas+clima · reg · 2026-06-01 listo (134 s)


## 3. Vista rápida

Solo para revisar los modelos base. La comparación con los ensambles se hace en 0416.

In [0]:
display(tabla_metricas(pd.concat([pred_ref, pred_lin]), ["version", "tarea", "modelo"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,Lineal,587957,0.0709,0.2852,0.7256,0.0586,0.2253,null,null,null,null,null,null
internas,clf,Referencia,587957,0.0709,0.0734,0.5179,0.0661,0.2575,null,null,null,null,null,null
internas,reg,Lineal,587951,null,null,null,null,null,7.9381,37.0563,11.5641,43.7136,102.0471,0.9655
internas,reg,Referencia,587951,null,null,null,null,null,8.1205,37.1139,10.4653,43.7282,112.5218,0.9655
internas+clima,clf,Lineal,587957,0.0709,0.3069,0.731,0.0576,0.2221,null,null,null,null,null,null
internas+clima,clf,Referencia,587957,0.0709,0.0734,0.5179,0.0661,0.2575,null,null,null,null,null,null
internas+clima,reg,Lineal,587951,null,null,null,null,null,8.0264,37.0332,11.5905,43.4171,105.653,0.9655
internas+clima,reg,Referencia,587951,null,null,null,null,null,8.1205,37.1139,10.4653,43.7282,112.5218,0.9655


In [0]:
display(tabla_metricas(pd.concat([pred_ref, pred_lin]), ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,Lineal,2026-04-01,196990,0.064,0.2595,0.7272,0.0539,0.2102,null,null,null,null,null,null
internas,clf,Lineal,2026-05-01,205304,0.0742,0.3124,0.7327,0.0601,0.2298,null,null,null,null,null,null
internas,clf,Lineal,2026-06-01,185663,0.0746,0.2791,0.7135,0.0619,0.2363,null,null,null,null,null,null
internas,clf,Referencia,2026-04-01,196990,0.064,0.064,0.5,0.06,0.2387,null,null,null,null,null,null
internas,clf,Referencia,2026-05-01,205304,0.0742,0.0742,0.5,0.069,0.2669,null,null,null,null,null,null
internas,clf,Referencia,2026-06-01,185663,0.0746,0.0746,0.5,0.0692,0.2669,null,null,null,null,null,null
internas,reg,Lineal,2026-04-01,196989,null,null,null,null,null,7.097,37.4752,9.7151,37.7444,97.9304,0.967
internas,reg,Lineal,2026-05-01,205301,null,null,null,null,null,7.8548,34.3574,12.5101,42.6898,99.907,0.9646
internas,reg,Lineal,2026-06-01,185661,null,null,null,null,null,8.9228,39.4104,12.4501,51.1969,108.7899,0.965
internas,reg,Referencia,2026-04-01,196989,null,null,null,null,null,7.3926,37.5013,8.6293,37.5388,112.5311,0.967


## 4. Guardar para 0416

In [0]:
pred_ref.to_parquet(f"{CARPETA_MODELADO}pred_val_Referencia.parquet", index=False)
pred_lin.to_parquet(f"{CARPETA_MODELADO}pred_val_Lineal.parquet", index=False)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_10/V3/MODELADO/
